# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
parquet_files


['../../05_src/data/prices/FLIC/FLIC_1995/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1995/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2014/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2014/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1993/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1993/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2020/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2020/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2018/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2018/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2011/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2011/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2001/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2001/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2008/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2008/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1988/part.0.parquet

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.
import shutil
# 'TEMP_DATA=../../05_src/data/temp/'
temp = os.getenv('TEMP_DATA')
temp_parquet = os.path.join(temp, 'parquet_hw')
shutil.rmtree(temp_parquet, ignore_errors=True)
os.makedirs(temp_parquet, exist_ok=True)

# for parquet_file in parquet_files[0:1]:
files = dd.read_parquet(parquet_files)

def add_features(pdf):
    pdf = pdf.sort_values(['ticker', 'Date']).copy()
    
    pdf['Close_lag_1'] = pdf.groupby('ticker')['Close'].shift(1)
    pdf['Adj_Close_lag_1'] = pdf.groupby('ticker')['Adj Close'].shift(1)
    pdf['returns'] = (pdf['Close'] / pdf['Close_lag_1']) - 1
    pdf['hi_lo_range'] = pdf['High'] - pdf['Low']

    return pdf

meta = files._meta.copy()

meta['Close_lag_1'] = 'float64'
meta['Adj_Close_lag_1'] = 'float64'
meta['returns'] = 'float64'
meta['hi_lo_range'] = 'float64'

dd_feat = files.map_partitions(add_features, meta=meta)

# dd_feat = (
#     file
#         .groupby('ticker', group_keys=True)
#         .apply(
#             lambda x: x.sort_values('Date', ascending=True)
#             .assign(close_lag_1=x['Close'].shift(1),
#                     Adj_close_lag_1=x['Adj Close'].shift(1),
#                     Returns=x['Close']/x['Close_lag_1']-1,
#                     hi_lo_range=x['High']-x['Low']),
#             meta = pd.DataFrame(date={'Date': 'datetime64[]'})
#             )
#     #     .assign(
#     # Close_lag_1 = lambda x: x['Close'].shift(1),
#     # Adj_close_lag_1 = lambda x: x['Adj Close'].shift(1),
#     # Returns = lambda x: x['Close'] / x['Close_lag_1'] - 1,
#     # hi_lo_range = lambda x: x['High'] - x['Low']
# )
# src = dd_feat['source'].compute().unique()
# tck = dd_feat['ticker'].compute().unique()
# yr = dd_feat['Year'].compute().unique()
# temp_path = os.path.join(temp_parquet, *src, *tck, str(*yr))
# dd_feat.to_parquet(temp_path, engine='pyarrow')

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.
import pandas as pd

df_dd = dd_feat.compute()
df_dd = df_dd.sort_values(['ticker', 'Date'])
df_dd['moving_average_10_days'] = (
    df_dd
    .groupby('ticker', group_keys=False)['Returns']
    .apply(lambda x:x.rolling(10).mean())
)
df_dd

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it wasn't necessary to convert to pandas to calculate the moving average return.
Yes, it would have been better to do it in Dask. Because it provides opportunity to handle more number of files at once.

In [ ]:
df_dd = df_dd.sort_values(['ticker', 'Date'])
dd_feat_copy = dd_feat.copy()
dd_feat_copy['moving_average_10_days'] = (
    df_dd
    .groupby('ticker')['Returns']
    .rolling(10)
    .mean()
    .reset_index(level=0, drop=True)
)

df_dd_copy = dd_feat_copy.compute()
df_dd_copy

But with Dask, rolling windows are trickier than pandas because the data is split into partitions. If a ticker's rows are split across partitions, Dask may not correctly calculate the rolling window across partition boundaries unless the dataframe is properly partitioned/sorted.

A safe Dask version is to apply the rolling calculation per ticker group

In [ ]:
def add_moving_average(pdf):
    pdf = pdf.sort_values('Date').copy()
    pdf['moving_average_10_days'] = pdf['returns'].rolling(10).mean()
    return pdf

meta = dd_feat._meta.copy()
meta['moving_average_10_days'] = 'float64'

dd_feat_ma = (
    dd_feat
    .groupby('ticker', group_keys=False)
    .apply(add_moving_average, meta=meta)
)

df_dd = dd_feat_ma.compute()
df_dd

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.